# Fine-tune Llama 3.2 for Price Prediction (cwait / Igniters)

**Week 7 – Community contribution:** Fine-tune **Meta Llama 3.2 1B** on a **small subset** of Week 7 prompt data using **Hugging Face** (transformers, datasets, peft, trl) and **QLoRA** to predict product prices from descriptions.

- **Model:** `meta-llama/Llama-3.2-1B`
- **Task:** "What does this cost to the nearest dollar?" + product summary → "Price is $X.00"
- **Data:** `ed-donner/items_prompts_lite` (small subset for quick runs)
- **Hardware:** Single GPU (e.g. Colab T4) with 4-bit QLoRA

## 1. Install dependencies

In [2]:
# Run locally or in Colab if needed (skip if your env already has these)
%pip install -q transformers datasets peft trl bitsandbytes accelerate

/Users/collinewaitire/projects/python-p/llm_engineering/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and config

In [3]:
import os
import re
from datetime import datetime
from getpass import getpass

import torch
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

In [5]:
# ── Model & data (cwait / Igniters – keep small for laptop-friendly runs) ──
BASE_MODEL = "meta-llama/Llama-3.2-1B"
DATASET_NAME = "ed-donner/items_prompts_lite"
SMALL_TRAIN_SIZE = 1500
VAL_SIZE = 200
EVAL_SIZE = 200

# ── QLoRA ──
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# ── Training ──
MAX_SEQUENCE_LENGTH = 128
EPOCHS = 1
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
LR_SCHEDULER_TYPE = "cosine"
SAVE_STEPS = 100
LOG_STEPS = 10

RUN_NAME = f"cwait-llama32-price-{datetime.now():%Y-%m-%d_%H.%M.%S}"
OUTPUT_DIR = RUN_NAME

try:
    capability = torch.cuda.get_device_capability()
    use_bf16 = capability[0] >= 8
except (AssertionError, RuntimeError):
    use_bf16 = False
    print("No CUDA GPU detected. use_bf16=False (fp16 used when you run on GPU).")
    print("Training requires a GPU (e.g. Colab T4).")

No CUDA GPU detected. use_bf16=False (fp16 used when you run on GPU).
Training requires a GPU (e.g. Colab T4).


## 3. Log in to Hugging Face

In [6]:
hf_token = os.environ.get("HF_TOKEN") or getpass("Hugging Face token: ")
login(token=hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 4. Load and prepare dataset (small subset)

In [7]:
dataset = load_dataset(DATASET_NAME)

train_raw = dataset["train"].select(range(min(SMALL_TRAIN_SIZE, len(dataset["train"]))))
val_raw = dataset["val"].select(range(min(VAL_SIZE, len(dataset["val"]))))

print(f"Train size: {len(train_raw)}, Val size: {len(val_raw)}")

Train size: 1500, Val size: 200


In [8]:
def add_text(example):
    example["text"] = example["prompt"] + example["completion"]
    return example

train_ds = train_raw.map(add_text, remove_columns=["prompt", "completion"])
val_ds = val_raw.map(add_text, remove_columns=["prompt", "completion"])

print("Sample:", train_ds[0]["text"][:200], "...")

Sample: What does this cost to the nearest dollar?

Title: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  
Category: Home Hardware  
Brand: Schlage  
Description: A single‑piece oil‑rubbed bronz ...


## 5. Load tokenizer and model (4-bit QLoRA)

In [9]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Model memory footprint: {model.get_memory_footprint() / 1e6:.1f} MB")

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 70f0949c-0fab-44a7-9205-1b10c5aea837)')' thrown while requesting HEAD https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/generation_config.json
Retrying in 1s [Retry 1/5].


generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Model memory footprint: 1012.0 MB


## 6. LoRA config and SFTTrainer

In [10]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_length=MAX_SEQUENCE_LENGTH,
    save_steps=SAVE_STEPS,
    logging_steps=LOG_STEPS,
    save_total_limit=2,
    optim="paged_adamw_32bit",
    report_to="none",
    run_name=RUN_NAME,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

## 7. Train

In [11]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.
/Users/collinewaitire/projects/python-p/llm_engineering/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


W0310 20:58:38.094000 86989 torch/_inductor/codegen/mps.py:173] [0/3_1] float64 cast requested, probably from tensorify_python_scalars
W0310 20:58:38.100000 86989 torch/_inductor/codegen/mps.py:173] [0/3_1] float64 cast requested, probably from tensorify_python_scalars
W0310 20:58:38.101000 86989 torch/_inductor/codegen/mps.py:173] [0/3_1] float64 cast requested, probably from tensorify_python_scalars
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the enviro

TrainOutput(global_step=94, training_loss=3.454816980564848, metrics={'train_runtime': 2063.0443, 'train_samples_per_second': 0.727, 'train_steps_per_second': 0.046, 'total_flos': 1071764682670080.0, 'train_loss': 3.454816980564848})

## 8. Save adapter

In [12]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

Saved to cwait-llama32-price-2026-03-10_20.51.56


## 9. Optional: Evaluate with Week 7 util

In [13]:
# Add week7 to path so we can import util (run from week7/community_contributions/cwait/ or copy util.py here)
import sys
WEEK7_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if os.path.isfile(os.path.join(WEEK7_ROOT, "util.py")) and WEEK7_ROOT not in sys.path:
    sys.path.insert(0, WEEK7_ROOT)

from util import evaluate

In [14]:
# Load test set as list of dicts (prompt, completion) for util.evaluate
test_raw = dataset["test"].select(range(min(EVAL_SIZE, len(dataset["test"]))))
test_list = [{"prompt": r["prompt"], "completion": r["completion"]} for r in test_raw]

# Parse completion to float for Tester (it expects float(datapoint["completion"]))
def _parse_completion(s):
    s = str(s).replace("$", "").replace(",", "").strip()
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0

for d in test_list:
    d["completion"] = _parse_completion(d["completion"])

print(f"Test list size: {len(test_list)}")

Test list size: 200


In [15]:
# Load base + adapter for inference
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
model_inference = PeftModel.from_pretrained(base, OUTPUT_DIR)
model_inference.eval()

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 57e60d60-ff80-46db-81ca-db143057d909)')' thrown while requesting HEAD https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/generation_config.json
Retrying in 1s [Retry 1/5].


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [16]:
def model_predict(datapoint):
    prompt = datapoint["prompt"]
    inputs = tokenizer(prompt, return_tensors="pt").to(model_inference.device)
    with torch.no_grad():
        out = model_inference.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

evaluate(model_predict, test_list, size=min(EVAL_SIZE, len(test_list)))